# 08 Stacking Ensemble 多因子选股

## 论文参考
- **作者**: Zengji Zheng
- **年份**: 2024
- **标题**: *A Comparative Study of Linear and Nonlinear Machine Learning Algorithms in Quantitative Investment*
- **DOI**: 10.54254/2755-2721/79/20241648
- **摘要**: 本文系统比较了线性与非线性机器学习算法在量化投资中的表现，并提出Stacking集成方法。策略以LightGBM、XGBoost和Ridge回归作为第一层基学习器，使用K-Fold交叉验证生成out-of-fold预测，再以Logistic回归作为第二层元学习器融合。实验证明Stacking集成在收益稳定性和风险调整收益上优于单一模型。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### Stacking (堆叠泛化)

Stacking通过分层学习融合多个模型的预测能力:

#### Layer 1: 基学习器 (Base Learners)
使用K-Fold交叉验证生成out-of-fold (OOF) 预测，避免信息泄漏:

$$\text{For each fold } k = 1, \ldots, K:$$
$$h_m^{(-k)} = \text{Train}(\mathcal{D} \setminus \mathcal{D}_k) \quad \text{(在第k折之外训练)}$$
$$\hat{z}_{i,m} = h_m^{(-k)}(x_i), \quad \forall i \in \mathcal{D}_k \quad \text{(对第k折样本预测)}$$

**基学习器**:
1. **LightGBM**: 基于直方图的梯度提升，高效处理大规模数据
2. **XGBoost**: 正则化梯度提升，防止过拟合
3. **Ridge回归**: L2正则化线性模型，提供线性基准

#### Layer 2: 元学习器 (Meta-Learner)
将OOF预测作为新特征，训练Logistic回归:
$$P(r_i > 0) = \sigma(w_0 + w_1 \hat{z}_{i,1} + w_2 \hat{z}_{i,2} + w_3 \hat{z}_{i,3})$$

其中 $\sigma(x) = \frac{1}{1 + e^{-x}}$ 为sigmoid函数

#### 选股策略
1. 将收益率预测转为分类问题 (收益>0 vs <=0)
2. Layer 1: 3个基模型分别用5-Fold CV生成OOF概率预测
3. Layer 2: Logistic回归融合3个概率
4. 按融合概率排序，选Top 10

In [ ]:
# === Cell 3: Stacking数据流动画 ===
import plotly.graph_objects as go
import numpy as np

# Create a data flow diagram showing stacking architecture
fig = go.Figure()

# Layer positions
input_x, l1_x, oof_x, l2_x, output_x = 0, 2.5, 5, 7.5, 10

# Colors
colors = {
    'input': '#607D8B', 'lgb': '#4CAF50', 'xgb': '#2196F3',
    'ridge': '#FF9800', 'meta': '#9C27B0', 'output': '#F44336'
}

# Draw boxes
boxes = [
    (input_x, 0, 'input', '15 Factors\n(Input)', 1.5),
    (l1_x, 2, 'lgb', 'LightGBM', 0.8),
    (l1_x, 0, 'xgb', 'XGBoost', 0.8),
    (l1_x, -2, 'ridge', 'Ridge', 0.8),
    (oof_x, 2, 'lgb', 'OOF Pred 1', 0.6),
    (oof_x, 0, 'xgb', 'OOF Pred 2', 0.6),
    (oof_x, -2, 'ridge', 'OOF Pred 3', 0.6),
    (l2_x, 0, 'meta', 'Logistic\nRegression', 1.0),
    (output_x, 0, 'output', 'Final\nScore', 0.8),
]

for (x, y, key, text, h) in boxes:
    fig.add_shape(type='rect',
        x0=x-0.8, x1=x+0.8, y0=y-h/2, y1=y+h/2,
        fillcolor=colors[key], opacity=0.8,
        line=dict(color='white', width=2))
    fig.add_annotation(x=x, y=y, text=text, showarrow=False,
        font=dict(color='white', size=11, family='Arial Black'))

# Draw arrows with animation frames
arrows = [
    # Input -> Base learners
    (input_x+0.8, 0, l1_x-0.8, 2), (input_x+0.8, 0, l1_x-0.8, 0),
    (input_x+0.8, 0, l1_x-0.8, -2),
    # Base -> OOF
    (l1_x+0.8, 2, oof_x-0.8, 2), (l1_x+0.8, 0, oof_x-0.8, 0),
    (l1_x+0.8, -2, oof_x-0.8, -2),
    # OOF -> Meta
    (oof_x+0.8, 2, l2_x-0.8, 0.3), (oof_x+0.8, 0, l2_x-0.8, 0),
    (oof_x+0.8, -2, l2_x-0.8, -0.3),
    # Meta -> Output
    (l2_x+0.8, 0, output_x-0.8, 0),
]

# Build frames showing data flow step by step
stages = [
    (0, 3, 'Stage 1: Input -> Base Learners'),
    (3, 6, 'Stage 2: 5-Fold CV -> OOF Predictions'),
    (6, 9, 'Stage 3: OOF -> Meta Learner'),
    (9, 10, 'Stage 4: Meta -> Final Score'),
]

frames = []
for stage_idx, (start, end, stage_name) in enumerate(stages):
    frame_shapes = []
    for a_idx in range(end):
        ax0, ay0, ax1, ay1 = arrows[a_idx]
        fig.add_annotation(
            x=ax1, y=ay1, ax=ax0, ay=ay0,
            arrowhead=3, arrowsize=1.5, arrowwidth=2,
            arrowcolor='#333' if a_idx < start else '#FF5722',
            opacity=1.0 if a_idx < end else 0.2,
        )

# Labels
fig.add_annotation(x=input_x, y=-3.5, text='输入层', font=dict(size=13, color='gray'), showarrow=False)
fig.add_annotation(x=l1_x, y=-3.5, text='Layer 1: 基学习器', font=dict(size=13, color='gray'), showarrow=False)
fig.add_annotation(x=oof_x, y=-3.5, text='OOF预测', font=dict(size=13, color='gray'), showarrow=False)
fig.add_annotation(x=l2_x, y=-3.5, text='Layer 2: 元学习器', font=dict(size=13, color='gray'), showarrow=False)
fig.add_annotation(x=output_x, y=-3.5, text='输出', font=dict(size=13, color='gray'), showarrow=False)

# K-Fold annotation
fig.add_annotation(x=(l1_x+oof_x)/2, y=3.2,
    text='5-Fold Cross Validation (防止信息泄漏)',
    font=dict(size=12, color='#F44336'), showarrow=False,
    bordercolor='#F44336', borderwidth=1, borderpad=4)

fig.update_layout(
    title=dict(text='Stacking Ensemble 架构: LightGBM + XGBoost + Ridge -> Logistic Regression'),
    xaxis=dict(visible=False, range=[-1.5, 11.5]),
    yaxis=dict(visible=False, range=[-4.5, 4.5]),
    height=550, width=1100, plot_bgcolor='white',
)
fig.show()

In [ ]:
# === Cell 4: 导入与配置 ===
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("LightGBM not installed, using GradientBoostingClassifier as fallback")

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed, using GradientBoostingClassifier as fallback")

if not HAS_LGB or not HAS_XGB:
    from sklearn.ensemble import GradientBoostingClassifier

from shared.data_fetcher import get_stock_daily, get_index_daily, get_multiple_stocks_daily
from shared.factors import build_factor_panel, winsorize, standardize
from shared.backtest_engine import MultiStockBacktester
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, plot_cumulative_comparison)
from shared.constants import *

print("All imports successful.")

In [ ]:
# === Cell 5: 数据获取 ===
STOCK_POOL = [
    "600519", "601318", "600036", "000858", "601166",
    "600276", "601398", "600030", "000333", "002415",
    "600900", "601888", "600809", "000568", "002304",
    "601012", "600031", "603259", "600585", "000661",
]

stock_data = get_multiple_stocks_daily(STOCK_POOL, DEFAULT_START, DEFAULT_END)
benchmark = get_index_daily(CSI300_CODE, DEFAULT_START, DEFAULT_END)

print(f"Successfully loaded {len(stock_data)} stocks")
print(f"Benchmark: {len(benchmark)} trading days")

In [ ]:
# === Cell 6: 因子工程 ===
panel = build_factor_panel(stock_data)
print(f"Factor panel shape: {panel.shape}")

FEATURE_COLS = [
    'mom_5', 'mom_10', 'mom_20', 'mom_60',
    'vol_5', 'vol_10', 'vol_20',
    'price_to_ma_5', 'price_to_ma_10', 'price_to_ma_20',
    'rsi', 'macd_hist', 'bb_pctb', 'bb_width', 'vp_corr',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in panel.columns]
TARGET_COL = 'fwd_return_20d'

for col in FEATURE_COLS:
    panel[col] = panel.groupby('date')[col].transform(
        lambda x: standardize(winsorize(x))
    )

panel.dropna(subset=FEATURE_COLS + [TARGET_COL], inplace=True)

# Binary classification target: return > 0
panel['target_cls'] = (panel[TARGET_COL] > 0).astype(int)

panel['date'] = pd.to_datetime(panel['date'])
panel['year_month'] = panel['date'].dt.to_period('M')
print(f"After cleaning: {panel.shape}")
print(f"Positive rate: {panel['target_cls'].mean():.2%}")

In [ ]:
# === Cell 7: Stacking训练 ===
months = sorted(panel['year_month'].unique())
TRAIN_WINDOW = 6
TOP_N = 10
N_FOLDS = 5

# Model definitions
def get_base_models():
    models = {}
    if HAS_LGB:
        models['LightGBM'] = lgb.LGBMClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            min_child_samples=10, verbose=-1, random_state=42
        )
    else:
        models['GBM_1'] = GradientBoostingClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            subsample=0.8, random_state=42
        )

    if HAS_XGB:
        models['XGBoost'] = xgb.XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric='logloss', verbosity=0, random_state=42
        )
    else:
        models['GBM_2'] = GradientBoostingClassifier(
            n_estimators=80, max_depth=4, learning_rate=0.1,
            subsample=0.8, random_state=43
        )

    models['Ridge'] = Ridge(alpha=1.0)
    return models

# Stacking selections and individual model selections for comparison
selections_stacking = {}
selections_individual = {name: {} for name in get_base_models()}
feature_importance_sum = None

for i in range(TRAIN_WINDOW, len(months) - 1):
    train_months = months[i - TRAIN_WINDOW:i]
    pred_month = months[i]

    train_df = panel[panel['year_month'].isin(train_months)]
    pred_df = panel[panel['year_month'] == pred_month]

    if len(train_df) < 50 or len(pred_df) < 5:
        continue

    X_train = train_df[FEATURE_COLS].values
    y_train = train_df['target_cls'].values
    y_train_reg = train_df[TARGET_COL].values  # for Ridge regression
    X_pred = pred_df[FEATURE_COLS].values

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_pred_s = scaler.transform(X_pred)

    base_models = get_base_models()

    # === Layer 1: Generate OOF predictions via K-Fold ===
    oof_train = np.zeros((len(X_train), len(base_models)))
    oof_pred = np.zeros((len(X_pred), len(base_models)))

    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

    for m_idx, (name, model) in enumerate(base_models.items()):
        try:
            fold_preds_test = np.zeros((len(X_pred), N_FOLDS))

            for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X_train_s)):
                X_tr, X_val = X_train_s[tr_idx], X_train_s[val_idx]

                if name == 'Ridge':
                    # Ridge uses regression, then convert to probability-like score
                    y_tr = y_train_reg[tr_idx]
                    from sklearn.base import clone
                    m = clone(model)
                    m.fit(X_tr, y_tr)
                    oof_train[val_idx, m_idx] = m.predict(X_val)
                    fold_preds_test[:, fold_idx] = m.predict(X_pred_s)
                else:
                    y_tr = y_train[tr_idx]
                    from sklearn.base import clone
                    m = clone(model)
                    m.fit(X_tr, y_tr)
                    if hasattr(m, 'predict_proba'):
                        oof_train[val_idx, m_idx] = m.predict_proba(X_val)[:, 1]
                        fold_preds_test[:, fold_idx] = m.predict_proba(X_pred_s)[:, 1]
                    else:
                        oof_train[val_idx, m_idx] = m.predict(X_val)
                        fold_preds_test[:, fold_idx] = m.predict(X_pred_s)

                    # Collect feature importance from first tree model
                    if fold_idx == 0 and m_idx == 0 and hasattr(m, 'feature_importances_'):
                        if feature_importance_sum is None:
                            feature_importance_sum = m.feature_importances_.copy()
                        else:
                            feature_importance_sum += m.feature_importances_

            oof_pred[:, m_idx] = fold_preds_test.mean(axis=1)

            # Individual model selection
            pred_df_copy = pred_df.copy()
            pred_df_copy['score'] = oof_pred[:, m_idx]
            top = pred_df_copy.nlargest(TOP_N, 'score')['symbol'].tolist()
            rebal_date = pred_df_copy['date'].max()
            selections_individual[name][rebal_date] = top

        except Exception as e:
            continue

    # === Layer 2: Meta-learner (Logistic Regression) ===
    try:
        meta_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
        # Handle Ridge output: normalize to [0,1] range
        oof_train_meta = oof_train.copy()
        for col_idx in range(oof_train_meta.shape[1]):
            col_min, col_max = oof_train_meta[:, col_idx].min(), oof_train_meta[:, col_idx].max()
            if col_max > col_min:
                oof_train_meta[:, col_idx] = (oof_train_meta[:, col_idx] - col_min) / (col_max - col_min)

        oof_pred_meta = oof_pred.copy()
        for col_idx in range(oof_pred_meta.shape[1]):
            col_min, col_max = oof_train[:, col_idx].min(), oof_train[:, col_idx].max()
            if col_max > col_min:
                oof_pred_meta[:, col_idx] = (oof_pred[:, col_idx] - col_min) / (col_max - col_min)

        meta_model.fit(oof_train_meta, y_train)
        stacking_proba = meta_model.predict_proba(oof_pred_meta)[:, 1]

        pred_df_copy = pred_df.copy()
        pred_df_copy['stacking_score'] = stacking_proba
        top_stocks = pred_df_copy.nlargest(TOP_N, 'stacking_score')['symbol'].tolist()

        rebal_date = pred_df_copy['date'].max()
        selections_stacking[rebal_date] = top_stocks

    except Exception as e:
        continue

print(f"Stacking: {len(selections_stacking)} rebalance periods")
for name in selections_individual:
    print(f"{name}: {len(selections_individual[name])} rebalance periods")

In [ ]:
# === Cell 8: 回测 ===
all_prices = {sym: stock_data[sym]['close'] for sym in stock_data}
bench_close = benchmark['close'] if 'close' in benchmark.columns else None

# Backtest stacking
bt = MultiStockBacktester(initial_capital=INITIAL_CAPITAL, rebalance_freq='M')
result_stacking = bt.run(all_prices, selections_stacking, bench_close)

print("=== Stacking Ensemble ===")
for k, v in result_stacking['metrics'].items():
    print(f"  {k}: {v}")

# Backtest individual models
results_individual = {}
for name in selections_individual:
    bt = MultiStockBacktester(initial_capital=INITIAL_CAPITAL, rebalance_freq='M')
    result = bt.run(all_prices, selections_individual[name], bench_close)
    results_individual[name] = result
    print(f"\n=== {name} ===")
    for k, v in result['metrics'].items():
        print(f"  {k}: {v}")

In [ ]:
# === Cell 9: 可视化 ===
import matplotlib.pyplot as plt
from shared.visualization import plot_factor_importance

# 1. All models cumulative returns comparison
all_returns = {'Stacking': result_stacking['returns']}
for name in results_individual:
    all_returns[name] = results_individual[name]['returns']
plot_cumulative_comparison(all_returns, title="Stacking vs 基学习器 累计收益对比")

# 2. Stacking equity curve vs benchmark
bench_eq = None
if result_stacking.get('benchmark_returns') is not None:
    bench_eq = INITIAL_CAPITAL * (1 + result_stacking['benchmark_returns']).cumprod()
plot_equity_curve(result_stacking['equity_curve'], bench_eq,
                  title="Stacking Ensemble vs 沪深300")

# 3. Drawdown
plot_drawdown(result_stacking['equity_curve'], title="Stacking Ensemble 回撤")

# 4. Performance comparison table
fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('off')
model_names = ['Stacking'] + list(results_individual.keys())
all_results = {'Stacking': result_stacking, **results_individual}
metrics_keys = ['累计收益率', '年化收益率', '夏普比率', '最大回撤', '胜率', 'Calmar比率']
cell_text = []
for key in metrics_keys:
    row = [key] + [all_results[m]['metrics'].get(key, 'N/A') for m in model_names]
    cell_text.append(row)
table = ax.table(cellText=cell_text,
                 colLabels=['指标'] + model_names,
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)
for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_facecolor('#2196F3')
        cell.set_text_props(color='white', fontweight='bold')
    elif j == 1 and i > 0:  # Highlight Stacking column
        cell.set_facecolor('#E3F2FD')
    elif i % 2 == 0:
        cell.set_facecolor('#f0f0f0')
ax.set_title("Stacking vs 单模型 绩效对比", fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# 5. Feature importance (from first tree model)
if feature_importance_sum is not None:
    importance_dict = dict(zip(FEATURE_COLS, feature_importance_sum))
    plot_factor_importance(importance_dict, title="基学习器因子重要性 (累计)", top_n=15)

## 结果讨论

### Stacking集成的优势
1. **模型多样性**: LightGBM擅长处理类别特征和高维数据，XGBoost具有强正则化能力，Ridge提供线性基准
2. **OOF防泄漏**: K-Fold交叉验证确保元学习器的训练数据不包含直接拟合信息
3. **自适应权重**: Logistic回归自动学习各基模型的最优融合权重

### Stacking vs 单模型
- Stacking通常在收益率稳定性上优于单一模型
- 最大回撤往往小于最激进的单模型
- 夏普比率反映风险调整后的优势

### 实现细节
- **K的选择**: K=5是常见选择，K太小则OOF预测方差大，K太大则计算开销增加
- **元学习器**: 使用简单的Logistic回归避免二次过拟合；不建议使用复杂模型
- **基模型多样性**: 线性+非线性组合效果最佳；相似模型的融合收益递减

### 与论文对比
- Zheng (2024) 证明Stacking在线性与非线性算法比较中表现稳健
- 集成方法的核心价值在于减少单一模型的过拟合风险
- 实际应用中可增加更多异质基模型 (如SVM、Neural Network)